In [1]:
import json, logging, numpy as np
from pathlib import Path
from datetime import datetime
from collections import Counter
 
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator
 
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)
 
print("torch   :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

torch   : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM    : 102.0 GB


In [2]:
DATA_DIR   = Path("/kaggle/input/datasets/tranquanghuy2809/data-embedding/data")
OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)
 
CFG = {
    "base_model":  "/kaggle/input/datasets/tranquanghuy2809/data-embedding/Vietnamese_Embedding_v2",
    "max_seq_len": 512,
    "mrl_dims":    [256, 512, 1024],
    "stages": [
        {"name": "stage1", "file": DATA_DIR / "train_stage1.jsonl", "epochs": 2, "lr": 2e-6, "batch": 128, "warmup": 50},
        {"name": "stage2", "file": DATA_DIR / "train_stage2.jsonl", "epochs": 2, "lr": 1e-6, "batch": 64, "warmup": 10},
        {"name": "stage3", "file": DATA_DIR / "train_stage3.jsonl", "epochs": 1, "lr": 5e-7, "batch": 64, "warmup": 5},
    ],
    "eval_file": DATA_DIR / "test_dataset.jsonl",
    "use_amp":   True,
}
 
print("Config OK")
print("MRL dims :", CFG["mrl_dims"])
print("Stages   :", [s["name"] for s in CFG["stages"]])

Config OK
MRL dims : [256, 512, 1024]
Stages   : ['stage1', 'stage2', 'stage3']


In [3]:
def load_stage_examples(path):
    examples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            q   = rec.get("query",    "").strip()
            pos = rec.get("positive", "").strip()
            if q and pos:
                examples.append(InputExample(texts=[q, pos])) 
    log.info(f"  {Path(path).name}: {len(examples)} examples")
    return examples
 
 
def build_evaluator(eval_path):
    """
    test_dataset.jsonl (master format) → InformationRetrievalEvaluator
    Corpus = gold chunks + mined negatives làm distractors
    """
    queries, corpus, relevant_docs = {}, {}, {}
 
    with open(eval_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            idx = rec["index"]
            qid = f"q{idx:04d}"
 
            queries[qid]       = rec["query"]
            relevant_docs[qid] = set()
 
            for pos in rec.get("positives", []):
                corpus[pos["chunk_id"]] = pos["text"]
                relevant_docs[qid].add(pos["chunk_id"])
 
            for neg in rec.get("negatives", []):
                if neg.get("type") == "mined" and neg.get("text", "").strip():
                    cid = f"neg_{idx}_{abs(hash(neg['text'])) % 1_000_000:06d}"
                    corpus[cid] = neg["text"]
 
    valid_q   = {k: v for k, v in queries.items()       if relevant_docs.get(k)}
    valid_rel = {k: v for k, v in relevant_docs.items() if v}
 
    log.info(f"Evaluator: {len(valid_q)} queries | {len(corpus)} corpus chunks")
 
    return InformationRetrievalEvaluator(
        queries           = valid_q,
        corpus            = corpus,
        relevant_docs     = valid_rel,
        accuracy_at_k     = [1, 3, 5, 10],
        mrr_at_k          = [10],
        batch_size        = 128,
        name              = "vn_embed",
        show_progress_bar = True,
        write_csv         = True,
    )
 
 
def run_stage(model, stage_cfg, evaluator, output_dir):
    name   = stage_cfg["name"]
    epochs = stage_cfg["epochs"]
    lr     = stage_cfg["lr"]
    bs     = stage_cfg["batch"]
    warmup = stage_cfg["warmup"]
 
    log.info(f"\n{'='*55}")
    log.info(f"STAGE: {name.upper()}  |  epochs={epochs}  lr={lr}  batch={bs}")
    log.info(f"{'='*55}")
 
    examples = load_stage_examples(stage_cfg["file"])
    loader   = DataLoader(examples, batch_size=bs, shuffle=True)
 
    mnr_loss = MultipleNegativesRankingLoss(
        model         = model,
        scale         = 20.0,
        # hardness_mode = "hard_negatives",
    )
    mrl_loss = MatryoshkaLoss(
        model           = model,
        loss            = mnr_loss,
        matryoshka_dims = CFG["mrl_dims"],
    )
 
    ckpt = str(output_dir / "checkpoints" / name)
 
    model.fit(
        train_objectives            = [(loader, mrl_loss)],
        evaluator                   = evaluator,
        epochs                      = epochs,
        warmup_steps                = warmup,
        optimizer_params            = {"lr": lr},
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        use_amp                     = CFG["use_amp"] and torch.cuda.is_available(),
        evaluation_steps            = len(loader),
        output_path                 = ckpt,
        save_best_model             = True,
        show_progress_bar           = True,
        # checkpoint_path             = ckpt,
        # checkpoint_save_steps       = len(loader),
        # checkpoint_save_total_limit = 2,
    )
    log.info(f"Stage {name} xong → {ckpt}")
 
 
print("Functions loaded OK")

Functions loaded OK


In [6]:
model = SentenceTransformer(CFG["base_model"])
model.max_seq_length = CFG["max_seq_len"]
log.info(f"Model loaded | dim={model.get_sentence_embedding_dimension()}")
 
evaluator = build_evaluator(CFG["eval_file"])
 
log.info("\n--- Baseline (trước khi train) ---")
evaluator(model, output_path=str(OUTPUT_DIR))

05:34:17 | Use pytorch device_name: cuda:0
05:34:17 | Load pretrained SentenceTransformer: /kaggle/input/datasets/tranquanghuy2809/data-embedding/Vietnamese_Embedding_v2


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

05:34:18 | Model loaded | dim=1024
05:34:18 | Evaluator: 157 queries | 2093 corpus chunks
05:34:18 | 
--- Baseline (trước khi train) ---
05:34:18 | Information Retrieval Evaluation of the model on the vn_embed dataset:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:10<00:00, 10.95s/it]
05:34:30 | Queries: 157
05:34:30 | Corpus: 2093

05:34:30 | Score-Function: cosine
05:34:30 | Accuracy@1: 50.32%
05:34:30 | Accuracy@3: 78.98%
05:34:30 | Accuracy@5: 87.26%
05:34:30 | Accuracy@10: 91.72%
05:34:30 | Precision@1: 50.32%
05:34:30 | Precision@3: 33.12%
05:34:30 | Precision@5: 24.71%
05:34:30 | Precision@10: 14.65%
05:34:30 | Recall@1: 32.77%
05:34:30 | Recall@3: 59.06%
05:34:30 | Recall@5: 71.13%
05:34:30 | Recall@10: 80.82%
05:34:30 | MRR@10: 0.6649
05:34:30 | NDCG@10: 0.6502
05:34:30 | MAP@100: 0.5705


{'vn_embed_cosine_accuracy@1': 0.5031847133757962,
 'vn_embed_cosine_accuracy@3': 0.7898089171974523,
 'vn_embed_cosine_accuracy@5': 0.8726114649681529,
 'vn_embed_cosine_accuracy@10': 0.9171974522292994,
 'vn_embed_cosine_precision@1': 0.5031847133757962,
 'vn_embed_cosine_precision@3': 0.33121019108280253,
 'vn_embed_cosine_precision@5': 0.2471337579617834,
 'vn_embed_cosine_precision@10': 0.1464968152866242,
 'vn_embed_cosine_recall@1': 0.3277070063694268,
 'vn_embed_cosine_recall@3': 0.590552016985138,
 'vn_embed_cosine_recall@5': 0.7112526539278131,
 'vn_embed_cosine_recall@10': 0.8081740976645435,
 'vn_embed_cosine_ndcg@10': 0.6501912390002795,
 'vn_embed_cosine_mrr@10': 0.664887271256698,
 'vn_embed_cosine_map@100': 0.5705379272263508}

In [7]:
run_stage(model, CFG["stages"][0], evaluator, OUTPUT_DIR)

05:34:51 | 
05:34:51 | STAGE: STAGE1  |  epochs=2  lr=2e-06  batch=128
05:34:51 | =======================================================
05:34:51 |   train_stage1.jsonl: 1041 examples


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Vn Embed Cosine Accuracy@1,Vn Embed Cosine Accuracy@3,Vn Embed Cosine Accuracy@5,Vn Embed Cosine Accuracy@10,Vn Embed Cosine Precision@1,Vn Embed Cosine Precision@3,Vn Embed Cosine Precision@5,Vn Embed Cosine Precision@10,Vn Embed Cosine Recall@1,Vn Embed Cosine Recall@3,Vn Embed Cosine Recall@5,Vn Embed Cosine Recall@10,Vn Embed Cosine Ndcg@10,Vn Embed Cosine Mrr@10,Vn Embed Cosine Map@100
9,No log,No log,0.509554,0.789809,0.872611,0.917197,0.509554,0.331210,0.247134,0.147134,0.329830,0.590552,0.711253,0.811359,0.652377,0.668072,0.571963
18,No log,No log,0.541401,0.802548,0.885350,0.923567,0.541401,0.335456,0.249682,0.148408,0.341720,0.600106,0.719745,0.819321,0.662688,0.687180,0.581448


05:35:01 | Information Retrieval Evaluation of the model on the vn_embed dataset in epoch 1.0 after 9 steps:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

05:36:58 | Information Retrieval Evaluation of the model on the vn_embed dataset in epoch 2.0 after 18 steps:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:03<00:00,  3.20s/it]
05:37:01 | Queries: 157
05:37:01 | Corpus: 2093

05:37:01 | Score-Function: cosine
05:37:01 | Accuracy@1: 54.14%
05:37:01 | Accuracy@3: 80.25%
05:37:01 | Accuracy@5: 88.54%
05:37:01 | Accuracy@10: 92.36%
05:37:01 | Precision@1: 54.14%
05:37:01 | Precision@3: 33.55%
05:37:01 | Precision@5: 24.97%
05:37:01 | Precision@10: 14.84%
05:37:01 | Recall@1: 34.17%
05:37:01 | Recall@3: 60.01%
05:37:01 | Recall@5: 71.97%
05:37:01 | Recall@10: 81.93%
05:37:01 | MRR@10: 0.6872
05:37:01 | NDCG@10: 0.6627
05:37:01 | MAP@100: 0.5814
05:37:01 | Stage stage1 xong → /kaggle/working/output/checkpoints/stage1


In [8]:
import gc

# Giải phóng memory từ stage 1
del model
gc.collect()
torch.cuda.empty_cache()

print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

# Load lại từ best stage1
model = SentenceTransformer("/kaggle/working/output/checkpoints/stage1")
model.max_seq_length = CFG["max_seq_len"]
log.info(f"Loaded stage1 best | dim={model.get_sentence_embedding_dimension()}")

05:38:25 | Use pytorch device_name: cuda:0
05:38:25 | Load pretrained SentenceTransformer: /kaggle/working/output/checkpoints/stage1


GPU free: 101.2 GB


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

05:38:27 | Loaded stage1 best | dim=1024


In [9]:
run_stage(model, CFG["stages"][1], evaluator, OUTPUT_DIR)

05:38:37 | 
05:38:37 | STAGE: STAGE2  |  epochs=2  lr=1e-06  batch=64
05:38:37 | =======================================================
05:38:37 |   train_stage2.jsonl: 1949 examples


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Vn Embed Cosine Accuracy@1,Vn Embed Cosine Accuracy@3,Vn Embed Cosine Accuracy@5,Vn Embed Cosine Accuracy@10,Vn Embed Cosine Precision@1,Vn Embed Cosine Precision@3,Vn Embed Cosine Precision@5,Vn Embed Cosine Precision@10,Vn Embed Cosine Recall@1,Vn Embed Cosine Recall@3,Vn Embed Cosine Recall@5,Vn Embed Cosine Recall@10,Vn Embed Cosine Ndcg@10,Vn Embed Cosine Mrr@10,Vn Embed Cosine Map@100
31,No log,No log,0.585987,0.834395,0.898089,0.949045,0.585987,0.354565,0.256051,0.151592,0.379406,0.642357,0.745223,0.842463,0.696180,0.723928,0.617970
62,No log,No log,0.592357,0.821656,0.904459,0.949045,0.592357,0.354565,0.252229,0.152866,0.376221,0.643737,0.741826,0.852017,0.699093,0.724222,0.618926


05:38:54 | Information Retrieval Evaluation of the model on the vn_embed dataset in epoch 1.0 after 31 steps:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

05:39:55 | Information Retrieval Evaluation of the model on the vn_embed dataset in epoch 2.0 after 62 steps:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:03<00:00,  3.21s/it]
05:39:58 | Queries: 157
05:39:58 | Corpus: 2093

05:39:58 | Score-Function: cosine
05:39:58 | Accuracy@1: 59.24%
05:39:58 | Accuracy@3: 82.17%
05:39:58 | Accuracy@5: 90.45%
05:39:58 | Accuracy@10: 94.90%
05:39:58 | Precision@1: 59.24%
05:39:58 | Precision@3: 35.46%
05:39:58 | Precision@5: 25.22%
05:39:58 | Precision@10: 15.29%
05:39:58 | Recall@1: 37.62%
05:39:58 | Recall@3: 64.37%
05:39:58 | Recall@5: 74.18%
05:39:58 | Recall@10: 85.20%
05:39:58 | MRR@10: 0.7242
05:39:58 | NDCG@10: 0.6991
05:39:58 | MAP@100: 0.6189
05:39:58 | Stage stage2 xong → /kaggle/working/output/checkpoints/stage2


In [10]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()

model = SentenceTransformer("/kaggle/working/output/checkpoints/stage2")
model.max_seq_length = CFG["max_seq_len"]
log.info(f"Loaded stage2 best | dim={model.get_sentence_embedding_dimension()}")

05:40:16 | Use pytorch device_name: cuda:0
05:40:16 | Load pretrained SentenceTransformer: /kaggle/working/output/checkpoints/stage2


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

05:40:18 | Loaded stage2 best | dim=1024


In [11]:
run_stage(model, CFG["stages"][2], evaluator, OUTPUT_DIR)

05:40:21 | 
05:40:21 | STAGE: STAGE3  |  epochs=1  lr=5e-07  batch=64
05:40:21 | =======================================================
05:40:21 |   train_stage3.jsonl: 5089 examples


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Vn Embed Cosine Accuracy@1,Vn Embed Cosine Accuracy@3,Vn Embed Cosine Accuracy@5,Vn Embed Cosine Accuracy@10,Vn Embed Cosine Precision@1,Vn Embed Cosine Precision@3,Vn Embed Cosine Precision@5,Vn Embed Cosine Precision@10,Vn Embed Cosine Recall@1,Vn Embed Cosine Recall@3,Vn Embed Cosine Recall@5,Vn Embed Cosine Recall@10,Vn Embed Cosine Ndcg@10,Vn Embed Cosine Mrr@10,Vn Embed Cosine Map@100
80,No log,No log,0.579618,0.847134,0.929936,0.955414,0.579618,0.363057,0.259873,0.155414,0.372293,0.657537,0.760934,0.859766,0.707021,0.727201,0.626588


05:41:05 | Information Retrieval Evaluation of the model on the vn_embed dataset in epoch 1.0 after 80 steps:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

05:41:41 | Information Retrieval Evaluation of the model on the vn_embed dataset in epoch 1.0 after 80 steps:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:03<00:00,  3.20s/it]
05:41:44 | Queries: 157
05:41:44 | Corpus: 2093

05:41:44 | Score-Function: cosine
05:41:44 | Accuracy@1: 57.96%
05:41:44 | Accuracy@3: 84.71%
05:41:44 | Accuracy@5: 92.99%
05:41:44 | Accuracy@10: 95.54%
05:41:44 | Precision@1: 57.96%
05:41:44 | Precision@3: 36.31%
05:41:44 | Precision@5: 25.99%
05:41:44 | Precision@10: 15.54%
05:41:44 | Recall@1: 37.23%
05:41:44 | Recall@3: 65.75%
05:41:44 | Recall@5: 76.09%
05:41:44 | Recall@10: 85.98%
05:41:44 | MRR@10: 0.7272
05:41:44 | NDCG@10: 0.7070
05:41:44 | MAP@100: 0.6266
05:41:44 | Stage stage3 xong → /kaggle/working/output/checkpoints/stage3


In [4]:
RETRIEVE_PATH = Path("/kaggle/input/datasets/tranquanghuy2809/data-embedding/retrieve_rerank_991.jsonl")
TEST_PATH     = Path("/kaggle/input/datasets/tranquanghuy2809/data-embedding/data/test_dataset.jsonl")

# Build corpus
corpus = {}
with open(RETRIEVE_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        rec = json.loads(line)
        for cand in rec.get("candidates", []):
            cid  = cand.get("chunk_id", "")
            text = cand.get("chunk", "").strip()
            if cid and text:
                corpus[cid] = text

print(f"Corpus: {len(corpus)} unique chunks")

# Build queries + gold
queries, relevant_docs = {}, {}
with open(TEST_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        rec = json.loads(line)
        qid = f"q{rec['index']:04d}"
        queries[qid]       = rec["query"]
        relevant_docs[qid] = {p["chunk_id"] for p in rec.get("positives", [])}

valid_q   = {k: v for k, v in queries.items()
             if relevant_docs.get(k) and relevant_docs[k] & corpus.keys()}
valid_rel = {k: relevant_docs[k] & corpus.keys() for k in valid_q}

print(f"Test queries     : {len(valid_q)}")
print(f"Có gold in corpus: {sum(1 for v in valid_rel.values() if v)}")

evaluator_full = InformationRetrievalEvaluator(
    queries=valid_q, corpus=corpus, relevant_docs=valid_rel,
    accuracy_at_k=[1,3,5,10], mrr_at_k=[10],
    batch_size=128, name="vn_embed_full", show_progress_bar=True,
)

Corpus: 2204 unique chunks
Test queries     : 157
Có gold in corpus: 157


In [14]:
import gc

final_path = Path("/kaggle/working/output/checkpoints/stage3")

finetuned = SentenceTransformer(str(final_path))
finetuned.max_seq_length = 512
print("\n--- Fine-tuned (sau train) ---")
evaluator_full(finetuned, output_path=str(OUTPUT_DIR))

05:42:30 | Use pytorch device_name: cuda:0
05:42:30 | Load pretrained SentenceTransformer: /kaggle/working/output/checkpoints/stage3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

05:42:31 | Information Retrieval Evaluation of the model on the vn_embed_full dataset:



--- Fine-tuned (sau train) ---


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:10<00:00, 10.69s/it]
05:42:42 | Queries: 157
05:42:42 | Corpus: 2204

05:42:42 | Score-Function: cosine
05:42:42 | Accuracy@1: 75.16%
05:42:42 | Accuracy@3: 91.72%
05:42:42 | Accuracy@5: 94.90%
05:42:42 | Accuracy@10: 97.45%
05:42:42 | Precision@1: 75.16%
05:42:42 | Precision@3: 43.95%
05:42:42 | Precision@5: 31.46%
05:42:42 | Precision@10: 17.32%
05:42:42 | Recall@1: 48.50%
05:42:42 | Recall@3: 74.92%
05:42:42 | Recall@5: 85.79%
05:42:42 | Recall@10: 93.21%
05:42:42 | MRR@10: 0.8339
05:42:42 | NDCG@10: 0.8165
05:42:42 | MAP@100: 0.7506


{'vn_embed_full_cosine_accuracy@1': 0.7515923566878981,
 'vn_embed_full_cosine_accuracy@3': 0.9171974522292994,
 'vn_embed_full_cosine_accuracy@5': 0.9490445859872612,
 'vn_embed_full_cosine_accuracy@10': 0.9745222929936306,
 'vn_embed_full_cosine_precision@1': 0.7515923566878981,
 'vn_embed_full_cosine_precision@3': 0.4394904458598726,
 'vn_embed_full_cosine_precision@5': 0.31464968152866246,
 'vn_embed_full_cosine_precision@10': 0.17324840764331212,
 'vn_embed_full_cosine_recall@1': 0.485031847133758,
 'vn_embed_full_cosine_recall@3': 0.7491507430997877,
 'vn_embed_full_cosine_recall@5': 0.8578556263269639,
 'vn_embed_full_cosine_recall@10': 0.9320594479830149,
 'vn_embed_full_cosine_ndcg@10': 0.8165011872775404,
 'vn_embed_full_cosine_mrr@10': 0.8339424729552118,
 'vn_embed_full_cosine_map@100': 0.7506173785584864}

In [15]:
import numpy as np

# Load fine-tuned model 1 lần
model_mrl = SentenceTransformer(str(final_path))
model_mrl.max_seq_length = 512

results = {}

for dim in [256, 512, 1024]:
    print(f"\n--- Testing dim={dim} ---")

    evaluator_dim = InformationRetrievalEvaluator(
        queries       = valid_q,
        corpus        = corpus,
        relevant_docs = valid_rel,
        accuracy_at_k = [1, 3, 5, 10],
        mrr_at_k      = [10],
        batch_size    = 128,
        name          = f"mrl_{dim}",
        show_progress_bar = True,
        truncate_dim  = dim,   # ← MRL truncation
    )

    scores = evaluator_dim(model_mrl, output_path=str(OUTPUT_DIR))
    results[dim] = scores

# Summary table
print("\n" + "="*60)
print(f"{'Dim':<8} {'Acc@1':>8} {'Acc@10':>8} {'MRR@10':>8} {'vs 1024':>10}")
print("="*60)
for dim in [256, 512, 1024]:
    s    = results[dim]
    acc1 = s.get(f"mrl_{dim}_cosine_accuracy@1", 0)
    acc10= s.get(f"mrl_{dim}_cosine_accuracy@10", 0)
    mrr  = s.get(f"mrl_{dim}_cosine_mrr@10", 0)
    ref  = results[1024].get(f"mrl_1024_cosine_accuracy@1", 0)
    delta= f"{(acc1-ref)*100:+.2f}%" if dim != 1024 else "baseline"
    print(f"{dim:<8} {acc1*100:>7.2f}% {acc10*100:>7.2f}% {mrr:>8.4f} {delta:>10}")
print("="*60)

05:43:06 | Use pytorch device_name: cuda:0
05:43:06 | Load pretrained SentenceTransformer: /kaggle/working/output/checkpoints/stage3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

05:43:07 | Information Retrieval Evaluation of the model on the mrl_256 dataset (truncated to 256):



--- Testing dim=256 ---


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:10<00:00, 10.80s/it]
05:43:40 | Queries: 157
05:43:40 | Corpus: 2204

05:43:40 | Score-Function: cosine
05:43:40 | Accuracy@1: 75.16%
05:43:40 | Accuracy@3: 91.72%
05:43:40 | Accuracy@5: 94.90%
05:43:40 | Accuracy@10: 97.45%
05:43:40 | Precision@1: 75.16%
05:43:40 | Precision@3: 43.95%
05:43:40 | Precision@5: 31.46%
05:43:40 | Precision@10: 17.32%
05:43:40 | Recall@1: 48.50%
05:43:40 | Recall@3: 74.92%
05:43:40 | Recall@5: 85.79%
05:43:40 | Recall@10: 93.21%
05:43:40 | MRR@10: 0.8339
05:43:40 | NDCG@10: 0.8165
05:43:40 | MAP@100: 0.7506



Dim         Acc@1   Acc@10   MRR@10    vs 1024
256        71.34%   96.18%   0.7953     -3.82%
512        77.71%   95.54%   0.8384     +2.55%
1024       75.16%   97.45%   0.8339   baseline


In [16]:
import shutil

# Tạo thư mục lưu
SAVE_DIR = Path("/kaggle/working/saved_models")
SAVE_DIR.mkdir(exist_ok=True)

# Lưu final model (stage3 best)
model_mrl.save(str(SAVE_DIR / "vn_embed_finetuned"))
print(f"✓ Saved: {SAVE_DIR / 'vn_embed_finetuned'}")

# Zip để download
shutil.make_archive(
    str(Path("/kaggle/working") / "vn_embed_finetuned"),
    "zip",
    str(SAVE_DIR / "vn_embed_finetuned"),
)
print(f"✓ Zipped: /kaggle/working/vn_embed_finetuned.zip")

# Kiểm tra size
zip_size = Path("/kaggle/working/vn_embed_finetuned.zip").stat().st_size
print(f"   Size: {zip_size/1e6:.1f} MB")

05:44:13 | Save model to /kaggle/working/saved_models/vn_embed_finetuned


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Saved: /kaggle/working/saved_models/vn_embed_finetuned
✓ Zipped: /kaggle/working/vn_embed_finetuned.zip
   Size: 1933.4 MB


In [5]:
BENCHMARK_MODELS = {
    "Vietnamese_Embedding_v1":       "/kaggle/input/datasets/tranquanghuy2809/data-embedding/Vietnamese_Embedding_v1",
    "Vietnamese_Embedding_v2":       "/kaggle/input/datasets/tranquanghuy2809/data-embedding/Vietnamese_Embedding_v2",
}

results = {}
for label, model_id in BENCHMARK_MODELS.items():
    print(f"\n--- {label} ---")
    import gc
    if 'bm' in dir(): del bm; gc.collect(); torch.cuda.empty_cache()
    
    bm = SentenceTransformer(model_id)
    bm.max_seq_length = 512
    scores = evaluator_full(bm, output_path=str(OUTPUT_DIR))
    results[label] = scores

results["VN_Embed_v2 Fine-tuned (512d)"] = {
    "vn_embed_full_cosine_accuracy@1": 0.7771,
    "vn_embed_full_cosine_mrr@10":     0.8384,
}

12:16:23 | Use pytorch device_name: cuda:0
12:16:23 | Load pretrained SentenceTransformer: /kaggle/input/datasets/tranquanghuy2809/data-embedding/Vietnamese_Embedding_v1



--- Vietnamese_Embedding_v1 ---


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

12:16:52 | Information Retrieval Evaluation of the model on the vn_embed_full dataset:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

12:17:22 | Information Retrieval Evaluation of the model on the vn_embed_full dataset:


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:10<00:00, 10.65s/it]
12:17:32 | Queries: 157
12:17:32 | Corpus: 2204

12:17:32 | Score-Function: cosine
12:17:32 | Accuracy@1: 71.34%
12:17:32 | Accuracy@3: 85.35%
12:17:32 | Accuracy@5: 90.45%
12:17:32 | Accuracy@10: 96.18%
12:17:32 | Precision@1: 71.34%
12:17:32 | Precision@3: 40.98%
12:17:32 | Precision@5: 28.92%
12:17:32 | Precision@10: 16.56%
12:17:32 | Recall@1: 45.27%
12:17:32 | Recall@3: 69.87%
12:17:32 | Recall@5: 79.76%
12:17:32 | Recall@10: 89.76%
12:17:32 | MRR@10: 0.7970
12:17:32 | NDCG@10: 0.7735
12:17:32 | MAP@100: 0.7062


In [6]:
results = {
    "Vietnamese_Embedding_v1": None,   
    "Vietnamese_Embedding_v2": None,   
}

if 'bm' in dir(): del bm; gc.collect(); torch.cuda.empty_cache()

bm = SentenceTransformer("/kaggle/input/datasets/tranquanghuy2809/data-embedding/vietnamese-bi-encoder")
bm.max_seq_length = 256
print("\n--- Vietnamese-bi-encoder (BKAI) ---")
scores = evaluator_full(bm, output_path=str(OUTPUT_DIR))
results["Vietnamese-bi-encoder (BKAI)"] = scores

12:17:33 | Use pytorch device_name: cuda:0
12:17:33 | Load pretrained SentenceTransformer: /kaggle/input/datasets/tranquanghuy2809/data-embedding/vietnamese-bi-encoder


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

12:17:36 | Information Retrieval Evaluation of the model on the vn_embed_full dataset:



--- Vietnamese-bi-encoder (BKAI) ---


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:02<00:00,  2.82s/it]
12:17:39 | Queries: 157
12:17:39 | Corpus: 2204

12:17:39 | Score-Function: cosine
12:17:39 | Accuracy@1: 56.05%
12:17:39 | Accuracy@3: 71.34%
12:17:39 | Accuracy@5: 75.80%
12:17:39 | Accuracy@10: 80.25%
12:17:39 | Precision@1: 56.05%
12:17:39 | Precision@3: 31.63%
12:17:39 | Precision@5: 22.42%
12:17:39 | Precision@10: 12.74%
12:17:39 | Recall@1: 37.02%
12:17:39 | Recall@3: 55.40%
12:17:39 | Recall@5: 62.77%
12:17:39 | Recall@10: 69.02%
12:17:39 | MRR@10: 0.6433
12:17:39 | NDCG@10: 0.6050
12:17:39 | MAP@100: 0.5519


In [7]:
import gc
del bm; gc.collect(); torch.cuda.empty_cache()

bm = SentenceTransformer("/kaggle/input/datasets/tranquanghuy2809/data-embedding/bge-m3")
bm.max_seq_length = 512
print("\n--- BGE-M3 (raw) ---")
scores = evaluator_full(bm, output_path=str(OUTPUT_DIR))
results["BGE-M3 (raw)"] = scores

12:17:39 | Use pytorch device_name: cuda:0
12:17:39 | Load pretrained SentenceTransformer: /kaggle/input/datasets/tranquanghuy2809/data-embedding/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

12:17:54 | Information Retrieval Evaluation of the model on the vn_embed_full dataset:



--- BGE-M3 (raw) ---


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:10<00:00, 10.60s/it]
12:18:04 | Queries: 157
12:18:04 | Corpus: 2204

12:18:04 | Score-Function: cosine
12:18:04 | Accuracy@1: 73.25%
12:18:04 | Accuracy@3: 91.08%
12:18:04 | Accuracy@5: 92.99%
12:18:04 | Accuracy@10: 97.45%
12:18:04 | Precision@1: 73.25%
12:18:04 | Precision@3: 44.80%
12:18:04 | Precision@5: 30.96%
12:18:04 | Precision@10: 17.39%
12:18:04 | Recall@1: 48.56%
12:18:04 | Recall@3: 77.12%
12:18:04 | Recall@5: 84.84%
12:18:04 | Recall@10: 93.74%
12:18:04 | MRR@10: 0.8240
12:18:04 | NDCG@10: 0.8194
12:18:04 | MAP@100: 0.7554
